In [1]:
import pandas as pd 
import numpy as np

In [2]:
df = pd.read_csv('data_collection_v11.csv')

/var/folders/lv/xlf_dnmj3svdvgpk2j8qrx9h0000gn/T/ipykernel_38372/3008515361.py:1: DtypeWarning: Columns (20,26,31,32,33,37,38,39,43,44,45,49,50,55,56,61,62,67,69,75,78,82,84,88,90,109,117,122,134,138,140,141,142,146,147,148,151,152,154,158,160,163,167,168,182,183,185,192,194,196,197,198,202,203,204,205,206,208,212,214,217,240,241,244,263,264,265,275,276,277) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('data_collection_v11.csv')


Table of Contents

- [Relocation type](#relocation-type)
- [Financial Assistance](#financial-assistance)
- [Bureau of Real Estate certification](#bureau-of-real-estate-certification)
    - [Relocation type: sponsor found or tenant found](#relocation-type-sponsor-found-or-tenant-found)
    - [Housing conditions on site and relocated](#housing-conditions-on-site-and-relocated)
- [Family relocation plan](#family-relocation-plan)
- [Race](#race)
- [Employment](#employment)
- [Monthly rent on-site](#monthly-rent-onsite)
- [Monthly rent relocated](#monthly-rent-relocated)

# Relocation type

In [3]:
df['reloc_type'].isna().value_counts(normalize=False)

False    1447
True      683
Name: reloc_type, dtype: int64

In [4]:
round(100*df['reloc_type'].isna().value_counts(normalize=True),1)

False    67.9
True     32.1
Name: reloc_type, dtype: float64

683 (or 32.9%) of all 2130 SORs do not have any information regarding the type of relocation, which was available in the back of the records. 

For the cards that had information in this section, it is important to note that multiple options could be checked. For example, a record may say that a tenant moved to "private rental housing" that was "tenant found".  

In [5]:
df['reloc_type'].value_counts()

Private Rental Housing, Tenant Found                        362
Private Rental Housing, Sponsor Found                       236
Tenant Found                                                198
Sponsor Found                                               195
Public Housing                                              162
Whereabouts Unknown                                          82
Tenant Found, Furnished Room                                 45
Purchased Home                                               32
Furnished Room                                               26
Sponsor Found, Public Housing                                24
Private Rental Housing                                       18
Tenant Found, Purchased Home                                 15
Private Rental Housing, Tenant Found, Sharing Apt.            8
Temporary Location                                            7
Sharing Apt.                                                  4
Tenant Found, Sharing Apt.              

In [6]:
df['reloc_type'].value_counts().sum() #should be 1447 records 

1447

Let's split this so we can count the number of SORs that had a category selected: 

In [7]:
df_relocation_type = df[['ID_long', 'reloc_type']]
df_relocation_type['reloc_type_split'] = df_relocation_type['reloc_type'].str.split(', ')
df_relocation_type_exploded = df_relocation_type.explode('reloc_type_split')

/var/folders/lv/xlf_dnmj3svdvgpk2j8qrx9h0000gn/T/ipykernel_38372/2764797071.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_relocation_type['reloc_type_split'] = df_relocation_type['reloc_type'].str.split(', ')


In [8]:
df_relocation_type_exploded['reloc_type_split'].value_counts(dropna=False)

NaN                       683
Tenant Found              649
Private Rental Housing    637
Sponsor Found             466
Public Housing            190
Whereabouts Unknown        84
Furnished Room             78
Purchased Home             54
Sharing Apt.               16
Temporary Location         11
hotel                       1
hotel                       1
Name: reloc_type_split, dtype: int64

In [9]:
df_relocation_type_exploded['reloc_type_split'].value_counts().sum()

2187

Among the 1447 records with relocation type, there are 2187 total answers for relocation type. 

This is how those answers are distributed: 

In [10]:
df_relocation_type_exploded['reloc_type_split'].value_counts()

Tenant Found              649
Private Rental Housing    637
Sponsor Found             466
Public Housing            190
Whereabouts Unknown        84
Furnished Room             78
Purchased Home             54
Sharing Apt.               16
Temporary Location         11
hotel                       1
hotel                       1
Name: reloc_type_split, dtype: int64

In [11]:
round(100*df_relocation_type_exploded['reloc_type_split'].value_counts(normalize=True),1)

Tenant Found              29.7
Private Rental Housing    29.1
Sponsor Found             21.3
Public Housing             8.7
Whereabouts Unknown        3.8
Furnished Room             3.6
Purchased Home             2.5
Sharing Apt.               0.7
Temporary Location         0.5
hotel                      0.0
hotel                      0.0
Name: reloc_type_split, dtype: float64

649 records checked off "tenant found". 466 checked "sponsor found". 

Let's create boolean variables that label whether a SOR stated in relocation type that it was sponsor found or tenant found. 

In [12]:
# create new columns in df  

##sponsor found 
# get all long_ids from df_relocation_type_exploded where reloc_type_split is "sponsor found" 
sponsor_found_ids = df_relocation_type_exploded[df_relocation_type_exploded['reloc_type_split'] == 'Sponsor Found']['ID_long']
# create column in df if long_id is found in sponsor_found_ids ("sponsor_found_relocation")
df['sponsor_found_relocation'] = df['ID_long'].isin(sponsor_found_ids).astype(int)

##tenant found
# get all long_ids from df_relocation_type_exploded where reloc_type_split is "tenant found" 
tenant_found_ids = df_relocation_type_exploded[df_relocation_type_exploded['reloc_type_split'] == 'Tenant Found']['ID_long']
# create column in df if long_id is found in tenant_found_ids ("tenant_found_relocation")
df['tenant_found_relocation'] = df['ID_long'].isin(tenant_found_ids).astype(int)

# Financial Assistance

**Is missingness in this section's answers dependent on whether the SOR card is marked sponsor found or tenant found?**

First, let's look at the financial assistance section variables and their overall missingness. 

Variables: 
* cash payment
* finders fee
* moving expense
* painting
* rent
* total


In [13]:
s = '''pay_date
pay_amm
findfee_date
findfee_amm
move_date
move_amm
paint_date
paint_amm
rent_date
rent_amm
total'''

In [14]:
s = s.split('\n')

In [15]:
s_amm = ['total'] + [i for i in s if i.endswith('_amm')] #money amounts only

In [16]:
len(s)

11

In [17]:
df[s].notna().any(axis=1).astype(int).value_counts()

0    1919
1     211
dtype: int64

211 rows have at least 1 non-null value amongst the financial assistance section columns. 

In [18]:
df[s_amm].notna().any(axis=1).astype(int).value_counts()

0    1924
1     206
dtype: int64

206 rows have at least 1 non-null value amongst financial assistance $$$ amount columns. 

How many rows have at least n non-null values across the subset of columns s? 

In [19]:
result = {n: (df[s].notna().sum(axis=1) >= n).sum() for n in range(1, 12)}
result 

{1: 211, 2: 164, 3: 38, 4: 32, 5: 0, 6: 0, 7: 0, 8: 0, 9: 0, 10: 0, 11: 0}

> 211 records have at least 1 value filled out, 164 records have at least 2 , 38 have at least 3, etc. 

In [20]:
result_amounts = {n: (df[s_amm].notna().sum(axis=1) >= n).sum() for n in range(1, 12)}
result_amounts 

{1: 206, 2: 39, 3: 1, 4: 0, 5: 0, 6: 0, 7: 0, 8: 0, 9: 0, 10: 0, 11: 0}

> 206 records have at least 1 money amount, 39 records have at least 2 money amounts, 1 record has at least 3 money amounts. 

Look at columns individually: 

In [21]:
for i in s:
    print(i)
    print(df[i].notna().astype(int).value_counts(), '\n')

# if 1, the variable was NOT missing

pay_date
0    2030
1     100
Name: pay_date, dtype: int64 

pay_amm
0    2027
1     103
Name: pay_amm, dtype: int64 

findfee_date
0    2129
1       1
Name: findfee_date, dtype: int64 

findfee_amm
0    2128
1       2
Name: findfee_amm, dtype: int64 

move_date
0    2032
1      98
Name: move_date, dtype: int64 

move_amm
0    1992
1     138
Name: move_amm, dtype: int64 

paint_date
0    2130
Name: paint_date, dtype: int64 

paint_amm
0    2130
Name: paint_amm, dtype: int64 

rent_date
0    2130
Name: rent_date, dtype: int64 

rent_amm
0    2130
Name: rent_amm, dtype: int64 

total
0    2127
1       3
Name: total, dtype: int64 



> 100 SORs had non-null cash payment date, 103 had non-null cash payment amount, etc. 

"Sparsity" ratio: 

> Of all values, what percentage are null? 

In [22]:
sparsity_ratio = df[s].isna().sum().sum() / df[s].size
print(f"Sparsity Ratio: {sparsity_ratio:.2%}")

Sparsity Ratio: 98.10%


Now, let's do crosstabs based on whether relocation type was sponsor found and then whether relocation type was tenant found. 


In [23]:
df[df['sponsor_found_relocation'] == 1][s].notna().any(axis=1).astype(int).value_counts()

0    411
1     55
dtype: int64

In [24]:
df[df['sponsor_found_relocation'] == 1][s_amm].notna().any(axis=1).astype(int).value_counts()

0    413
1     53
dtype: int64

> 55 records of the 466 where relocation type includes sponsor found have at least 1 non-null value amongst the financial assistance columns. 

In [25]:
#what percentage of sponsor found relocations are those? 

round(100*55/466,1)

11.8

In [26]:
df[df['tenant_found_relocation'] == 1][s].notna().any(axis=1).astype(int).value_counts()

0    549
1    100
dtype: int64

In [27]:
df[df['tenant_found_relocation'] == 1][s_amm].notna().any(axis=1).astype(int).value_counts()

0    549
1    100
dtype: int64

> 100 records of the 649 where relocation type includes tenant found have at least 1 non-null value amongst the financal assistance columns

In [28]:
# what percentage of tenant found relocations are those? 

round(100*100/649, 1)

15.4

In [29]:
466+649

1115

Are there any records with sponsor_found_relocation == 1 and tenant_found_relocation == 1? 

In [30]:
m = (df['sponsor_found_relocation'] == 1) & (df['tenant_found_relocation'] == 1)
m.astype(int).value_counts()

0    2124
1       6
dtype: int64

> THOSE SIX RECORDS **MAKE! NO! SENSE!**
> > Either tenant found or sponsor found. 
> > > Only six records though. 

In [31]:
df[m][s].notna().any(axis=1).astype(int).value_counts()

0    5
1    1
dtype: int64

In [32]:
df[m][s_amm].notna().any(axis=1).astype(int).value_counts()

0    5
1    1
dtype: int64

One of these six records has at least 1 non-null value amongst the columns in the financial assistance section. 

What about rate of missingness in those that aren't sponsor found and also aren't tenant found relocation? 

In [33]:
m = (df['sponsor_found_relocation'] == 0) & (df['tenant_found_relocation'] == 0)
m.astype(int).value_counts()

0    1109
1    1021
dtype: int64

1109 SORs satisfy ~m (`not m`). That's 1115 minus 6. THE WORLD MAKES SENSE. 

In [34]:
df[m][s].notna().any(axis=1).astype(int).value_counts()

0    964
1     57
dtype: int64

In [35]:
round(100*57/1021,1) #an overall lower rate of missingness

5.6

>11.8% of the sponsor found relocation records have at least one non-null answer in the set of all financial assistance variables. 

>15.4% of the tenant found relocation records have at least one non-null answer in the set of all financial assistance variables. 

>5.6% of records that are neither sponsor nor tenant found relocation records have at least one non-null answer in the set of all financial assistance variables. 

# Bureau of Real Estate certification

In [36]:
# the add_info and notes columns were inspected to see if the standalone token "bre" was in either of the columns 
# those records containing the token were further MANUALLY inspected to see if BRE certification was issued or not for a standard apartment. assumptions were made.
bre_cert_df = pd.read_csv('bre_cert - bre_cert.csv')
bre_cert_df = bre_cert_df[['ID_long', 'bre_cert']]

# create list of unique ID_longs where bre_cert = 1 to filter the dataframe for more analysis further down the line
bre_cert_id_list = bre_cert_df[bre_cert_df['bre_cert'] == 1]['ID_long'].tolist()

In [37]:
len(bre_cert_id_list)

304

In [38]:
# filter df to include only records that we assessed received a BRE certification of standard apartment
bre_cert_standard_df = df[df['ID_long'].isin(bre_cert_id_list)]

In [39]:
bre_cert_standard_df.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 304 entries, 3 to 2127
Columns: 282 entries, Unnamed: 0 to tenant_found_relocation
dtypes: float64(63), int64(6), object(213)
memory usage: 672.1+ KB


## Relocation type: sponsor found or tenant found

In [40]:
bre_cert_standard_df['sponsor_found_relocation'].value_counts()

1    160
0    144
Name: sponsor_found_relocation, dtype: int64

In [41]:
bre_cert_standard_df['tenant_found_relocation'].value_counts()

0    246
1     58
Name: tenant_found_relocation, dtype: int64

In [42]:
m1 = (bre_cert_standard_df['sponsor_found_relocation'] == 0) 
m2 = (bre_cert_standard_df['tenant_found_relocation'] == 0) 

In [43]:
bre_cert_standard_df[m1&m2].shape

(87, 282)

In [44]:
144+58+87

289

The discrepancy here (304 vs 289) has to do with different reloc_type values for the rows assumed to have a BRE certification of standard.

In [45]:
bre_cert_standard_df['reloc_type'].value_counts(dropna=False)

Sponsor Found                                           82
Private Rental Housing, Sponsor Found                   75
NaN                                                     70
Private Rental Housing, Tenant Found                    33
Tenant Found                                            16
Public Housing                                          13
Tenant Found, Furnished Room                             5
Purchased Home                                           3
Sponsor Found, Public Housing                            1
Private Rental Housing                                   1
Private Rental Housing, Tenant Found, Sharing Apt.       1
Private Rental Housing, Tenant Found, Purchased Home     1
Sponsor Found, Furnished Room                            1
Tenant Found, Sharing Apt.                               1
Private Rental Housing, Sponsor Found, Tenant Found      1
Name: reloc_type, dtype: int64

In [46]:
bre_cert_standard_df[~m1&~m2].shape 

(1, 282)

And there is one record with BRE cert OK that said it was both sponsor and tenant found. 

In [47]:
#70 SORs here have missing reloc_type, 13 have reloc type of public housing, 3 purchased home, 1 is private rental housing
70+13+3+1 

87

Matches `bre_cert_standard_df[m1&m2].shape` code result above. HURRAY! 

## Housing conditions on site and relocated

In [48]:
bre_cert_standard_df.shape

(304, 282)

**Bath and toilet** 

In [49]:
s = '''comp_bath_prvt_os
comp_bath_prvt_r
comp_bath_shrd_os
comp_bath_shrd_r
sep_toilet_prvt_os
sep_toilet_prvt_r
sep_toilet_shrd_os
sep_toilet_shrd_r
out_fac_prvt_os
out_fac_prvt_r
out_fac_shrd_os
out_fac_shrd_r'''

s = s.split('\n')

# split variables for on site and relocated 
s_os = [i for i in s if i.endswith('_os')]
s_r = [i for i in s if i.endswith('_r')]

In [50]:
df[s].notna().any(axis=1).astype(int).value_counts()

0    1225
1     905
dtype: int64

905 records had at least 1 non-null answer for the bath and toilet variables

In [51]:
905/(2130)

0.42488262910798125

In [52]:
#how many records had n or more answers for the variables in s? 

{n: (df[s].notna().sum(axis=1) >= n).sum() for n in range(1, len(s)+1)} 

{1: 905,
 2: 281,
 3: 92,
 4: 41,
 5: 0,
 6: 0,
 7: 0,
 8: 0,
 9: 0,
 10: 0,
 11: 0,
 12: 0}

In [53]:
sparsity_ratio = df[s].isna().sum().sum() / df[s].size
print(f"Sparsity Ratio: {sparsity_ratio:.2%}")

Sparsity Ratio: 94.84%


Instead of all SOR, what about those we determined had BRE certification of standard apartment? 

In [54]:
bre_cert_standard_df[s].notna().any(axis=1).astype(int).value_counts()

1    160
0    144
dtype: int64

In [55]:
160/(160+144)

0.5263157894736842

In [56]:
{n: (bre_cert_standard_df[s].notna().sum(axis=1) >= n).sum() for n in range(1, len(s)+1)}

{1: 160, 2: 39, 3: 17, 4: 3, 5: 0, 6: 0, 7: 0, 8: 0, 9: 0, 10: 0, 11: 0, 12: 0}

In [57]:
sparsity_ratio = bre_cert_standard_df[s].isna().sum().sum() / bre_cert_standard_df[s].size
print(f"Sparsity Ratio: {sparsity_ratio:.2%}")

Sparsity Ratio: 94.00%


bath and toilet @ on site: 

In [58]:
df[s_os].notna().any(axis=1).astype(int).value_counts()

0    1248
1     882
dtype: int64

In [59]:
{n: (df[s_os].notna().sum(axis=1) >= n).sum() for n in range(1, len(s_os)+1)}

{1: 882, 2: 125, 3: 0, 4: 0, 5: 0, 6: 0}

In [60]:
sparsity_ratio = df[s_os].isna().sum().sum() / df[s_os].size
print(f"Sparsity Ratio: {sparsity_ratio:.2%}")

Sparsity Ratio: 92.12%


In [61]:
bre_cert_standard_df[s_os].notna().any(axis=1).astype(int).value_counts()

1    155
0    149
dtype: int64

In [62]:
{n: (bre_cert_standard_df[s_os].notna().sum(axis=1) >= n).sum() for n in range(1, len(s_os)+1)}

{1: 155, 2: 22, 3: 0, 4: 0, 5: 0, 6: 0}

In [63]:
sparsity_ratio = bre_cert_standard_df[s_os].isna().sum().sum() / bre_cert_standard_df[s_os].size
print(f"Sparsity Ratio: {sparsity_ratio:.2%}")

Sparsity Ratio: 90.30%


bath and toilet @ relocation: 

In [64]:
df[s_r].notna().any(axis=1).astype(int).value_counts()

0    1902
1     228
dtype: int64

In [65]:
{n: (df[s_r].notna().sum(axis=1) >= n).sum() for n in range(1, len(s_r)+1)}

{1: 228, 2: 84, 3: 0, 4: 0, 5: 0, 6: 0}

In [66]:
sparsity_ratio = df[s_r].isna().sum().sum() / df[s_r].size
print(f"Sparsity Ratio: {sparsity_ratio:.2%}")

Sparsity Ratio: 97.56%


In [67]:
bre_cert_standard_df[s_r].notna().any(axis=1).astype(int).value_counts()

0    272
1     32
dtype: int64

In [68]:
{n: (bre_cert_standard_df[s_r].notna().sum(axis=1) >= n).sum() for n in range(1, len(s_r)+1)}

{1: 32, 2: 10, 3: 0, 4: 0, 5: 0, 6: 0}

In [69]:
sparsity_ratio = bre_cert_standard_df[s_r].isna().sum().sum() / bre_cert_standard_df[s_r].size
print(f"Sparsity Ratio: {sparsity_ratio:.2%}")

Sparsity Ratio: 97.70%


In [70]:
def get_number_of_answers_and_sparcity_ratio(DF, S):
    
    sparsity_ratio = DF[S].isna().sum().sum() / DF[S].size
    print(f"Sparsity Ratio: {sparsity_ratio:.2%}")

    results = {n: (DF[S].notna().sum(axis=1)>= n).sum() for n in range(1, len(S)+1)}
    print(f"Number of answers:", results)
    return sparsity_ratio, results

**Cooking & refrig**

(same thing for this set of variables)

In [71]:
s = '''range_os
range_r
coal_stv_os
no_cook_fac_os
no_cook_fac_r
refrig_os
icebx_os
icebx_r
norefrig_os
norefrig_r
coal_stc_r
refirg_r'''

s = s.split('\n')
s_os = [i for i in s if i.endswith('_os')]
s_r = [i for i in s if i.endswith('_r')]

In [72]:
len(s), len(s_r), len(s_os)

(12, 6, 6)

In [73]:
print("all records, all columns:")
_,_ = get_number_of_answers_and_sparcity_ratio(df, s)
print("\nall records, on site:")
_,_ = get_number_of_answers_and_sparcity_ratio(df, s_os) #on-site
print("\nall records, relocated:")
_,_ = get_number_of_answers_and_sparcity_ratio(df, s_r) #relocation
print("\n\nBRE cert, all columns:")
_,_ = get_number_of_answers_and_sparcity_ratio(bre_cert_standard_df, s)
print("\nBRE cert, on site:")
_,_ = get_number_of_answers_and_sparcity_ratio(bre_cert_standard_df, s_os) #on-site
print("\nBRE cert, relocated:")
_,_ = get_number_of_answers_and_sparcity_ratio(bre_cert_standard_df, s_r) #relocation

all records, all columns:
Sparsity Ratio: 91.79%
Number of answers: {1: 883, 2: 831, 3: 197, 4: 184, 5: 2, 6: 1, 7: 0, 8: 0, 9: 0, 10: 0, 11: 0, 12: 0}

all records, on site:
Sparsity Ratio: 87.09%
Number of answers: {1: 863, 2: 786, 3: 1, 4: 0, 5: 0, 6: 0}

all records, relocated:
Sparsity Ratio: 96.49%
Number of answers: {1: 233, 2: 212, 3: 2, 4: 1, 5: 0, 6: 0}


BRE cert, all columns:
Sparsity Ratio: 90.38%
Number of answers: {1: 156, 2: 146, 3: 25, 4: 24, 5: 0, 6: 0, 7: 0, 8: 0, 9: 0, 10: 0, 11: 0, 12: 0}

BRE cert, on site:
Sparsity Ratio: 83.83%
Number of answers: {1: 154, 2: 141, 3: 0, 4: 0, 5: 0, 6: 0}

BRE cert, relocated:
Sparsity Ratio: 96.93%
Number of answers: {1: 29, 2: 27, 3: 0, 4: 0, 5: 0, 6: 0}


**Heat & Hot Water**

(same thing for this set of variables)

In [74]:
s='''centr_heat_os
centr_heat_r
ind_heat_os
ind_heat_r
no_heat_os
no_heat_r
centr_hot_wtr_os
centr_hot_wtr_r
ind_hot_watr_os
ind_hot_watr_r
no_hot_watr_os
no_hot_watr_r'''

s = s.split('\n')
s_os = [i for i in s if i.endswith('_os')]
s_r = [i for i in s if i.endswith('_r')]

In [75]:
len(s), len(s_r), len(s_os)

(12, 6, 6)

In [76]:
print("all records, all columns:")
_,_ = get_number_of_answers_and_sparcity_ratio(df, s)
print("\nall records, on site:")
_,_ = get_number_of_answers_and_sparcity_ratio(df, s_os) #on-site
print("\nall records, relocated:")
_,_ = get_number_of_answers_and_sparcity_ratio(df, s_r) #relocation
print("\n\nBRE cert, all columns:")
_,_ = get_number_of_answers_and_sparcity_ratio(bre_cert_standard_df, s)
print("\nBRE cert, on site:")
_,_ = get_number_of_answers_and_sparcity_ratio(bre_cert_standard_df, s_os) #on-site
print("\nBRE cert, relocated:")
_,_ = get_number_of_answers_and_sparcity_ratio(bre_cert_standard_df, s_r) #relocation

all records, all columns:
Sparsity Ratio: 91.51%
Number of answers: {1: 902, 2: 846, 3: 218, 4: 200, 5: 3, 6: 0, 7: 0, 8: 0, 9: 0, 10: 0, 11: 0, 12: 0}

all records, on site:
Sparsity Ratio: 86.68%
Number of answers: {1: 885, 2: 810, 3: 7, 4: 0, 5: 0, 6: 0}

all records, relocated:
Sparsity Ratio: 96.35%
Number of answers: {1: 241, 2: 223, 3: 3, 4: 0, 5: 0, 6: 0}


BRE cert, all columns:
Sparsity Ratio: 90.08%
Number of answers: {1: 161, 2: 152, 3: 27, 4: 22, 5: 0, 6: 0, 7: 0, 8: 0, 9: 0, 10: 0, 11: 0, 12: 0}

BRE cert, on site:
Sparsity Ratio: 83.33%
Number of answers: {1: 158, 2: 145, 3: 1, 4: 0, 5: 0, 6: 0}

BRE cert, relocated:
Sparsity Ratio: 96.82%
Number of answers: {1: 31, 2: 26, 3: 1, 4: 0, 5: 0, 6: 0}


**Apartment characteristics: on-site vs relocated**

(same things for these set of variables)

In [77]:
s = '''grss_rent_os
grss_rent_r
no_room_os
no_room_r
overcrwd_os
overcrwd_r
int_room_os
int_room_r
adq_lght_os
adq_light_r
need_repair_os
need_repair_r'''

s = s.split('\n')
s_os = [i for i in s if i.endswith('_os')]
s_r = [i for i in s if i.endswith('_r')]

print(len(s), len(s_r), len(s_os))


12 6 6


In [78]:
print("all records, all columns:")
_,_ = get_number_of_answers_and_sparcity_ratio(df, s)
print("\nall records, on site:")
_,_ = get_number_of_answers_and_sparcity_ratio(df, s_os) #on-site
print("\nall records, relocated:")
_,_ = get_number_of_answers_and_sparcity_ratio(df, s_r) #relocation
print("\n\nBRE cert, all columns:")
_,_ = get_number_of_answers_and_sparcity_ratio(bre_cert_standard_df, s)
print("\nBRE cert, on site:")
_,_ = get_number_of_answers_and_sparcity_ratio(bre_cert_standard_df, s_os) #on-site
print("\nBRE cert, relocated:")
_,_ = get_number_of_answers_and_sparcity_ratio(bre_cert_standard_df, s_r) #relocation

all records, all columns:
Sparsity Ratio: 85.70%
Number of answers: {1: 625, 2: 579, 3: 469, 4: 442, 5: 371, 6: 257, 7: 177, 8: 170, 9: 155, 10: 153, 11: 143, 12: 114}

all records, on site:
Sparsity Ratio: 81.59%
Number of answers: {1: 598, 2: 532, 3: 382, 4: 344, 5: 317, 6: 180}

all records, relocated:
Sparsity Ratio: 89.81%
Number of answers: {1: 292, 2: 268, 3: 206, 4: 189, 5: 184, 6: 163}


BRE cert, all columns:
Sparsity Ratio: 85.12%
Number of answers: {1: 95, 2: 88, 3: 67, 4: 61, 5: 55, 6: 39, 7: 25, 8: 25, 9: 24, 10: 24, 11: 23, 12: 17}

BRE cert, on site:
Sparsity Ratio: 79.82%
Number of answers: {1: 89, 2: 79, 3: 62, 4: 58, 5: 50, 6: 30}

BRE cert, relocated:
Sparsity Ratio: 90.41%
Number of answers: {1: 35, 2: 32, 3: 30, 4: 28, 5: 26, 6: 24}


> These have more answers. Variables pertaining to relocated site are way more sparse than on-site for all records and BRE certified as standard records ...... 

# Family relocation plan 

In [79]:
s = '''room_req
rent_rng
area_des
hous_type'''
s = s.split('\n')

In [80]:
for i in s:
    print(i)
    print(df[i].isnull().astype(int).value_counts())
    print('\n')

room_req
0    1105
1    1025
Name: room_req, dtype: int64


rent_rng
1    1283
0     847
Name: rent_rng, dtype: int64


area_des
1    1279
0     851
Name: area_des, dtype: int64


hous_type
1    1128
0    1002
Name: hous_type, dtype: int64




Rooms required missing for 1025 records of 2130, desired rent rage missing for 1283 records of 2130, desired area missing for 1279 records of 2130, and housing type desired missing for 1128 records of 2130. 

In [81]:
_,_ = get_number_of_answers_and_sparcity_ratio(df, s)

Sparsity Ratio: 55.34%
Number of answers: {1: 1243, 2: 1054, 3: 873, 4: 635}


# Race 

In [82]:
race2_to_race3_dictionary = {'Puerto Rican':'Puerto Rican', 
                             'White':'White', 
                             'Oriental':'Oriental', 
                             'Puerto Rican mix':'Puerto Rican',
                             'other':'other',
                             'Negro':'Negro', 
                             'Multi':'Multi',
                             np.nan: np.nan}

df['race3'] = df['race2'].apply(lambda x: race2_to_race3_dictionary[x])

In [83]:
df['race3'].value_counts(dropna=False)

NaN             878
White           563
Puerto Rican    548
Negro            94
other            28
Oriental         15
Multi             4
Name: race3, dtype: int64

In [84]:
df['race3'].isna().astype(int).value_counts()

0    1252
1     878
Name: race3, dtype: int64

878 records had no answer or no discernible answer for race. 

# Employment

In [85]:
s='''emplyer
empl_addr'''
s = s.split('\n')

In [86]:
for i in s:
    print(i)
    print(df[i].isnull().astype(int).value_counts())
    print('\n')

emplyer
1    1696
0     434
Name: emplyer, dtype: int64


empl_addr
1    1732
0     398
Name: empl_addr, dtype: int64




In [87]:
_,_ = get_number_of_answers_and_sparcity_ratio(df, s)

Sparsity Ratio: 80.47%
Number of answers: {1: 511, 2: 321}


# Monthly rent on-site

In [88]:
df_addtl = pd.read_csv('data_v11_selected_variables.csv')

In [89]:
(df_addtl['rent_os_grss']/12).describe()

count    1768.000000
mean       46.860930
std        17.720189
min         0.000000
25%        34.372500
50%        43.333333
75%        57.095000
max       140.833333
Name: rent_os_grss, dtype: float64

In [90]:
df_addtl['rent_os_grss'].isna().astype(int).value_counts()

0    1768
1     362
Name: rent_os_grss, dtype: int64

362 records missing monthly on-site rent! 

# Monthly rent relocated 

In [91]:
df_addtl2 = pd.read_csv('data_v12.csv')

/var/folders/lv/xlf_dnmj3svdvgpk2j8qrx9h0000gn/T/ipykernel_38372/1655734138.py:1: DtypeWarning: Columns (19,25,30,31,32,36,37,38,42,43,44,48,49,54,55,60,61,66,68,74,77,81,83,87,89,108,116,121,133,137,139,140,141,145,146,147,150,151,153,157,159,162,166,167,181,182,184,191,193,195,196,197,201,202,203,204,205,207,211,213,216,239,240,243,262,263,264,274,275,276,295,296,304) have mixed types. Specify dtype option on import or set low_memory=False.
  df_addtl2 = pd.read_csv('data_v12.csv')


In [92]:
df_addtl2['rent_r_mth'].describe()

count    1381.000000
mean       66.116826
std        25.427402
min        19.450000
25%        49.350000
50%        60.000000
75%        78.000000
max       220.000000
Name: rent_r_mth, dtype: float64

In [93]:
df_addtl2['rent_r_mth'].isna().astype(int).value_counts()

0    1381
1     749
Name: rent_r_mth, dtype: int64

749 records missing monthly rent at relocation